# Ingest sprints.json file
1. Read the all the files from the sprints folder using spark dataframe reader API
1. Define and enforce schema 
1. Add Metadata Columns 
    - Source File
    - Ingestion Timestamp
1. Write to bronze delta table    

> Note: JSON is in multi line format

In [0]:
%run ../../../proyecto_final_formula1/PrepAmb/02.Ayuda_Bronze

In [0]:
%run ../../../proyecto_final_formula1/PrepAmb/03.Configuracion_Ambiente

In [0]:
# Define source_file and table_name
archivo_fuente = f"{ruta}/sprints"
nombre_tabla = f"{catalogo}.{esquema_bronze}.sprints"

#### Step 1 - Read the JSON file using the dataframe reader API

In [0]:
# Define the schema
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, FloatType, DateType

esquema_sprints = StructType([
    StructField("date", StringType()),
    StructField("raceName", StringType()),
    StructField("round", IntegerType()),
    StructField("season", IntegerType()),
    StructField("url", StringType()),
    StructField("constructorId", StringType()),
    StructField("driverId", StringType()),
    StructField("grid", IntegerType()),
    StructField("laps", IntegerType()),
    StructField("number", IntegerType()),
    StructField("points", FloatType()),
    StructField("position", IntegerType()),
    StructField("positionText", StringType()),
    StructField("status", StringType())
])


In [0]:
# Read data from the sprints file
df_sprints = (
    spark.read
       .format('json')
       .schema(esquema_sprints)
       .option('mode', 'FAILFAST')
       .option('multiLine', True)
       .load(archivo_fuente)
)

In [0]:
display(df_sprints)

#### Step 2 - Add Metadata Columns
- Source File
- Ingestion Timestamp

In [0]:
df_sprints_final = add_ingesta_metadata(df_sprints)

#### Step 3 - Write to bronze delta table

In [0]:
(
    df_sprints_final
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(nombre_tabla)
)

In [0]:
display(spark.table(nombre_tabla))